# Lab 10: Sectra Memory Analyst, Memory Forensics with Volatility 3 and an LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab10_llm_memory_forensics_volatility/memory_agent_jupyter.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  ·  **SectraTEK Lab**

Volatility already gives an examiner a list of plugins (`pslist`, `pstree`, `malfind`, ...). In this
lab, **that plugin list is the agent's toolset**. You ask a question in plain English; the model
picks one plugin and its arguments; our code runs Volatility; the model reads the output and either
runs another plugin or answers, **citing the plugin and the PID** it relied on.

The memory image is the public **Cridex** sample, a classic Windows XP banking-trojan infection with
documented ground truth, so you can check every answer the AI gives against the
[answer key](https://github.com/danjethh/sectratek-AI-Lab/blob/main/lab10_llm_memory_forensics_volatility/answer_key.md).

### 1. Install Volatility 3 and write the lab code

In [ ]:
%pip install -q volatility3==2.28.2 openai requests   # pinned: the answer key was verified on 2.28.2

In [ ]:
import os
os.makedirs('case', exist_ok=True)

In [ ]:
%%writefile agentkit.py
"""
agentkit.py - the shared forensic-agent core used by the Lab 07-10 notebooks and chatbots.

One place defines: how to connect to a model (Groq, OpenRouter or Ollama), how to build the
synthetic Android image, the read-only tools the agent may call, the think-act-observe
loop with its evidence check, and a conversational wrapper for the chatbots. The chatbot .py apps import this
module; the teaching notebooks embed the same code inline so students open only one file.
"""
import contextlib
import inspect
import json
import os
import re
import shutil
import sqlite3
import tempfile
from pathlib import Path


def as_text(content):
    """Message content as plain text. Gradio 6 can hand a chat message back as a list of parts
    ([{'type': 'text', 'text': ...}]) instead of a string; every caller goes through this."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join((x.get("text") or x.get("content") or "") if isinstance(x, dict) else str(x)
                       for x in content)
    if isinstance(content, dict):
        return str(content.get("text") or content.get("content") or "")
    return str(content or "")


@contextlib.contextmanager
def sqlite_copy(path):
    """Open a SQLite evidence file WITHOUT touching it: query a temporary copy (with its -wal and
    -journal files, so recent rows are included). Opening the original, even read-only, can create
    -shm/-wal files beside it, and a path with '#' or '?' can be misread as a different file."""
    path = Path(path)
    tmp = Path(tempfile.mkdtemp(prefix="sectra_db_"))
    try:
        shutil.copy2(path, tmp / "evidence.db")
        for side in ("-wal", "-journal"):
            src = path.with_name(path.name + side)
            if src.exists():
                shutil.copy2(src, tmp / ("evidence.db" + side))
        con = sqlite3.connect(tmp / "evidence.db")
        try:
            yield con
        finally:
            con.close()
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def read_head(path, n=16):
    """The first n bytes of a file (its 'magic'), without reading the whole file into memory."""
    with open(path, "rb") as f:
        return f.read(n)


# ======================================================================= model backend
# Free models that follow the agent's JSON format, best first. Providers retire models from
# time to time, so connect_model picks the first one your key can actually use, and the
# notebooks print which one it chose. Put your own choice first, or pass model="...".
GROQ_MODELS = ["openai/gpt-oss-120b", "qwen/qwen3.8-27b"]
OPENROUTER_MODELS = ["google/gemma-4-31b-it:free", "nvidia/nemotron-3-ultra-550b-a55b:free"]
NOT_CHAT = ("whisper", "orpheus", "guard", "tts", "embed")   # speech / safety / embedding models


def pick_model(client, preferred, free_only=False):
    """Return the first preferred model this key can use, else the largest chat model it lists."""
    try:
        listed = client.models.list().data
    except Exception:
        return preferred[0]          # cannot list models: try the first choice and let it report
    ids = [m.id for m in listed]
    for m in preferred:
        if m in ids:
            return m
    chat = [m for m in listed if not any(w in m.id.lower() for w in NOT_CHAT)
            and (m.id.endswith(":free") or not free_only)]
    chat.sort(key=lambda m: getattr(m, "context_window", 0) or 0, reverse=True)
    return chat[0].id if chat else preferred[0]


def connect_model(provider="groq", model=None, api_key=None, base_url=None):
    """Return (client, model_id). provider is 'groq' (default), 'openrouter' or 'ollama'.

    groq:       free hosted models, key from https://console.groq.com/keys (the same key as Labs 01-06).
    openrouter: free hosted models, key from https://openrouter.ai/keys.
    ollama:     a Qwen model running locally (https://ollama.com), no key, nothing leaves the PC.

    max_retries=8: the free tiers allow a few thousand tokens per minute. When a step goes over,
    the provider says "try again in N seconds" and the client waits and retries instead of failing.
    """
    from openai import OpenAI
    provider = provider.lower()
    if provider == "ollama":
        base_url = base_url or "http://localhost:11434/v1"
        client = OpenAI(base_url=base_url, api_key="ollama")  # Ollama ignores the key
        return client, (model or "qwen3:8b")
    if provider == "openrouter":
        client = OpenAI(base_url=base_url or "https://openrouter.ai/api/v1",
                        api_key=api_key or os.environ.get("OPENROUTER_API_KEY", ""), max_retries=8)
        return client, (model or pick_model(client, OPENROUTER_MODELS, free_only=True))
    # default: Groq
    client = OpenAI(base_url=base_url or "https://api.groq.com/openai/v1",
                    api_key=api_key or os.environ.get("GROQ_API_KEY", ""), max_retries=8)
    return client, (model or pick_model(client, GROQ_MODELS))


# ======================================================================= synthetic image
def build_image(root="android_image"):
    """Create a tiny, synthetic Android extraction on disk. Everything in it is fictional."""
    ROOT = Path(root)
    text_files = {
        f"{root}/build.prop":
            "# Synthetic build.prop (teaching data, fictional)\n"
            "ro.product.manufacturer=Google\nro.product.brand=google\n"
            "ro.product.model=Pixel 7a\nro.product.device=lynx\nro.product.name=lynx\n"
            "ro.build.version.release=14\nro.build.version.sdk=34\n"
            "ro.build.fingerprint=google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys\n",
        f"{root}/data/system/packages.list":
            "com.google.android.gm 10089 0 /data/user/0/com.google.android.gm default 0 1\n"
            "com.android.providers.contacts 10003 0 /data/user/0/com.android.providers.contacts platform 0 1\n"
            "com.android.providers.telephony 10004 0 /data/user/0/com.android.providers.telephony platform 0 1\n"
            "com.whatsapp 10137 0 /data/user/0/com.whatsapp default 0 1\n"
            "com.instagram.android 10140 0 /data/user/0/com.instagram.android default 0 1\n"
            "org.thoughtcrime.securesms 10142 0 /data/user/0/org.thoughtcrime.securesms default 0 1\n",
        f"{root}/data/system/users/0/settings_secure.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<settings version=\"241\">\n"
            "  <setting id=\"1\" name=\"android_id\" value=\"a1b2c3d4e5f60718\" package=\"android\" />\n"
            "  <setting id=\"2\" name=\"bluetooth_name\" value=\"Pixel 7a\" package=\"com.android.settings\" />\n"
            "  <setting id=\"3\" name=\"device_name\" value=\"Nadia's Pixel 7a\" package=\"com.android.settings\" />\n"
            "</settings>\n",
        f"{root}/data/misc/wifi/WifiConfigStore.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<WifiConfigStoreData>\n  <NetworkList>\n"
            "    <Network><string name=\"SSID\">&quot;HomeNet_5G&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721295600000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;BrightBeans Cafe&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1720950000000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;Okafor-Office&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721210400000\" /></Network>\n"
            "  </NetworkList>\n</WifiConfigStoreData>\n",
        # a JSON artifact (many Android apps store prefs/state as JSON)
        f"{root}/data/data/com.whatsapp/files/me.json":
            json.dumps({"jid": "15550143788@s.whatsapp.net", "pushname": "Nadia O",
                        "registered": True, "last_backup": "2024-07-18"}, indent=2),
    }
    for rel, content in text_files.items():
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content)

    def make_db(rel, schema, inserts):
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        if p.exists():
            p.unlink()
        con = sqlite3.connect(p)
        con.executescript(schema)
        for sql, params in inserts:
            con.executemany(sql, params)
        con.commit()
        con.close()

    make_db(f"{root}/data/data/com.google.android.gm/databases/accounts.db",
            "CREATE TABLE accounts(name TEXT, type TEXT, last_sync INTEGER);",
            [("INSERT INTO accounts VALUES (?,?,?)",
              [("nadia.okafor@gmail.com", "com.google", 1721295600),
               ("+15550143788", "com.whatsapp", 1721240000)])])
    make_db(f"{root}/data/data/com.android.providers.contacts/databases/contacts2.db",
            "CREATE TABLE contacts(display_name TEXT, phone TEXT, times_contacted INTEGER);",
            [("INSERT INTO contacts VALUES (?,?,?)",
              [("Marcus Webb", "+15550142210", 31), ("Priya Shah", "+15550148890", 12),
               ("Dad", "+15550149001", 54), ("BrightBeans Cafe", "+15550140077", 3),
               ("Alina K (work)", "+15550147325", 20)])])
    make_db(f"{root}/data/data/com.android.providers.telephony/databases/mmssms.db",
            "CREATE TABLE sms(address TEXT, date INTEGER, body TEXT, type INTEGER);",
            [("INSERT INTO sms VALUES (?,?,?,?)",
              [("+15550142210", 1721200000000, "Are we still on for Friday?", 1),
               ("+15550142210", 1721200600000, "Yes, 6pm at the usual place.", 2),
               ("+15550149001", 1721240500000, "Landed safe, call you tonight.", 2),
               ("+15550148890", 1721250000000, "Can you send the invoice?", 1),
               ("+15550149001", 1721299000000, "Thanks Dad, talk soon.", 2)])])
    make_db(f"{root}/data/data/com.whatsapp/databases/msgstore.db",
            "CREATE TABLE messages(chat TEXT, timestamp INTEGER, from_me INTEGER, text TEXT);",
            [("INSERT INTO messages VALUES (?,?,?,?)",
              [("Marcus Webb", 1721260000000, 0, "Did the package arrive?"),
               ("Marcus Webb", 1721260400000, 1, "Not yet, will check tomorrow."),
               ("Priya Shah", 1721280000000, 1, "Sent the files to your email.")])])
    return ROOT


# ======================================================================= the tools
# A text-ish file is one the agent can read as text. Everything else is "binary".
_TEXT_EXT = {".txt", ".xml", ".json", ".prop", ".list", ".log", ".csv", ".ini", ".conf", ".html"}


def make_tools(root="android_image", allow_python=False):
    """Build the read-only toolbox, all sealed inside `root`. Returns {name: function}."""
    ROOT = Path(root)

    def _resolve(path):
        base = ROOT.resolve()
        path = (path or ".").strip().lstrip("/")
        target = base if path in ("", ".") else (ROOT / path).resolve()
        if target != base and base not in target.parents:
            raise ValueError("path is outside the image")
        return target

    def list_dir(path="."):
        """List what is inside a folder (folders shown with a trailing '/')."""
        t = _resolve(path)
        if not t.exists():
            return f"No such path: {path}"
        if t.is_file():
            return f"{path} is a file, not a folder. Use read_file or run_sql."
        items = [e + ("/" if (t / e).is_dir() else "") for e in sorted(os.listdir(t))]
        return "\n".join(items) if items else "(empty folder)"

    def read_file(path, max_chars=1500):
        """Read a file. Pretty-prints JSON; for SQLite databases points you to run_sql;
        for other binary files shows the type and the first bytes."""
        t = _resolve(path)
        if not t.exists() or t.is_dir():
            return f"No such file: {path}"
        head = read_head(t)
        if head[:15] == b"SQLite format 3":
            with sqlite_copy(t) as con:
                tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
            return f"[{path} is a SQLite database. Tables: {tables}. Use run_sql with a SELECT query.]"
        ext = t.suffix.lower()
        if ext not in _TEXT_EXT and b"\x00" in head:
            return (f"[{path} is a binary file ({t.stat().st_size} bytes). First bytes (hex): "
                    f"{head.hex(' ')}. It is not readable as text.]")
        text = t.read_text(errors="replace")
        if ext == ".json":
            try:
                text = json.dumps(json.loads(text), indent=2)
            except Exception:
                pass
        return text[:max_chars] + ("" if len(text) <= max_chars else "\n...[truncated]")

    def run_sql(path, query):
        """Run ONE read-only SQL SELECT against a SQLite database and return the rows."""
        t = _resolve(path)
        if not t.exists():
            return f"No such database: {path}"
        q = (query or "").strip().rstrip(";")
        if not q.lower().startswith(("select", "pragma")):
            return "Only read-only SELECT queries are allowed."
        try:
            with sqlite_copy(t) as con:   # a temporary copy: the evidence file is never opened
                cur = con.execute(q)
                cols = [c[0] for c in cur.description] if cur.description else []
                rows = cur.fetchmany(50)
        except Exception as e:
            return f"SQL error: {e}"
        if not rows:
            return "(no rows)"
        return "\n".join([" | ".join(cols)] + [" | ".join(str(c) for c in r) for r in rows])

    def find_files(name):
        """Find files anywhere in the image whose path contains `name` (case-insensitive)."""
        base = ROOT.resolve()
        needle = (name or "").lower()
        hits = []
        for p in sorted(base.rglob("*")):
            if p.is_symlink():
                continue                      # never follow a link out of the image
            if p.is_file() and needle in str(p.relative_to(base)).lower():
                hits.append(str(p.relative_to(base)))
        if not hits:
            return f"No files matched {name!r}."
        return "\n".join(hits[:50]) + ("" if len(hits) <= 50 else f"\n...({len(hits)} total)")

    def search_text(keyword, max_hits=20):
        """Search the text inside readable files for `keyword` (case-insensitive).
        Returns path:line for each match, so you know which file to open next."""
        base = ROOT.resolve()
        needle = (keyword or "").lower()
        out = []
        for p in sorted(base.rglob("*")):
            if p.is_symlink() or not p.is_file() or p.suffix.lower() not in _TEXT_EXT:
                continue
            try:
                for i, line in enumerate(p.read_text(errors="replace").splitlines(), 1):
                    if needle in line.lower():
                        out.append(f"{p.relative_to(base)}:{i}: {line.strip()[:120]}")
                        if len(out) >= max_hits:
                            return "\n".join(out) + "\n...[more matches not shown]"
            except Exception:
                continue
        return "\n".join(out) if out else f"No text matches for {keyword!r}."

    tools = {"list_dir": list_dir, "read_file": read_file, "run_sql": run_sql,
             "find_files": find_files, "search_text": search_text}

    if allow_python:
        def run_python(code):
            """ADVANCED, OPTIONAL: run a short Python snippet to inspect the image. `ROOT` (the image
            folder, a pathlib.Path) is available. Print what you want to see. Do not write or delete."""
            # NOT a sandbox: the keyword check below stops accidents, not a determined snippet (Python
            # can still import modules and touch files). Use it only on a COPY of an extraction.
            import io
            import contextlib
            if any(bad in code for bad in ("open(", "write", "unlink", "remove", "rmtree",
                                           "os.system", "subprocess", "Popen", "__import__")):
                return "Blocked: that snippet looks like it writes or runs external commands."
            buf = io.StringIO()
            safe = {"ROOT": ROOT, "sqlite3": sqlite3, "json": json, "Path": Path, "re": re}
            try:
                with contextlib.redirect_stdout(buf):
                    exec(code, safe)  # teaching tool only; see the warning above
            except Exception as e:
                return f"Python error: {e}\n{buf.getvalue()}"
            out = buf.getvalue().strip()
            return out[:1500] if out else "(the snippet ran but printed nothing)"
        tools["run_python"] = run_python

    return tools


def tool_descriptions(tools):
    """One line per tool for the system prompt: its real signature (so the model uses the right
    argument names) and the first paragraph of its docstring."""
    lines = []
    for name, fn in tools.items():
        try:
            sig = str(inspect.signature(fn))
        except (TypeError, ValueError):
            sig = "()"
        doc = " ".join((fn.__doc__ or "").strip().split("\n\n")[0].split())
        lines.append(f"- {name}{sig}: {doc}")
    return "\n".join(lines)


# ======================================================================= the agent
def build_system_prompt(tools):
    return (
        "You are Sectra, a digital-forensics examiner's assistant. You are investigating an Android "
        "device extraction (a copy of the phone's files) to answer the examiner's question. You cannot "
        "see any file until you open it with a tool, and you must base every answer only on what the "
        "tools return.\n\n"
        "TOOLS (use exactly one per step):\n" + tool_descriptions(tools) + "\n\n"
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To use a tool:\n"
        '  {"thought": "<one short sentence>", "action": "list_dir", "action_input": {"path": "data/data"}}\n'
        "When you can answer, reply:\n"
        '  {"thought": "<one short sentence>", "final_answer": "<your answer>", "evidence_path": "<the file you used>"}\n\n'
        "RULES: investigate step by step; use ONLY what the tools return; never invent paths, tables, "
        "or data; if a tool errors, try another path or tool; if the evidence is not there, say so "
        "plainly; always cite the exact file in evidence_path. Keep thoughts to one sentence. /no_think"
    )


def parse_action(raw):
    """Pull the JSON object out of the model's reply, tolerating <think> blocks, code fences, and
    prose before or after it (the first '{' that starts a valid JSON object wins)."""
    text = re.sub(r"<think>.*?</think>", "", raw or "", flags=re.S)
    text = text.replace("```json", "").replace("```", "")
    decoder = json.JSONDecoder()
    for m in re.finditer(r"\{", text):
        try:
            obj, _ = decoder.raw_decode(text, m.start())
        except ValueError:
            continue
        if isinstance(obj, dict):
            return obj
    return None


def chat(client, model, messages, max_tokens=None):
    """One model call. Some hosted models (gpt-oss on Groq) occasionally try a built-in function call
    instead of writing the JSON reply, which the provider rejects ("model called a tool"); when that
    happens, ask again with a plain-text reminder instead of stopping the investigation.
    max_tokens: 1,500 by default, 900 for Qwen (its free tier allows 1,000 reply tokens a minute)."""
    if max_tokens is None:
        max_tokens = 900 if "qwen" in str(model).lower() else 1500
    for attempt in range(3):
        try:
            return client.chat.completions.create(model=model, messages=messages, temperature=0,
                                                  max_tokens=max_tokens).choices[0].message.content or ""
        except Exception as e:
            if "called a tool" not in str(e) or attempt == 2:
                raise
            messages = messages + [{"role": "user", "content": "Do not use function calling. Reply with "
                                    "ONE JSON object written as plain text."}]


def run_agent_iter(question, client, model, tools, system_prompt=None, max_steps=12, history=None):
    """Generator version of the agent, for live UIs. Yields a dict per step
    ({step, thought, action, args, observation}) and finally the answer
    ({final_answer, evidence_path}) or a give-up dict ({final_answer: None}).

    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    """
    system_prompt = system_prompt or build_system_prompt(tools)
    messages = [{"role": "system", "content": system_prompt}]
    for u, a in (history or []):
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    question = as_text(question)
    messages.append({"role": "user", "content": f"Investigate and answer this question: {question}"})

    seen = question + " " + " ".join(as_text(a) for _, a in (history or []))  # what it may rely on
    tools_run, pushbacks, ran = 0, 0, set()
    for step in range(1, max_steps + 1):
        raw = chat(client, model, messages)
        messages.append({"role": "assistant", "content": raw})
        move = parse_action(raw)
        if move is None:
            messages.append({"role": "user",
                             "content": "That was not valid JSON. Reply with ONE JSON object only."})
            continue
        if "final_answer" in move:
            answer = move["final_answer"]
            problem = evidence_check(answer, tools_run, seen, cited=move.get("evidence_path", ""),
                                     ran=ran, tool_names=list(tools))
            if problem and pushbacks < MAX_PUSHBACKS:
                # The harness, not the model, decides whether an answer is backed by evidence.
                pushbacks += 1
                yield {"step": step, "thought": "Evidence check: answer sent back",
                       "action": "evidence_check", "args": {}, "observation": problem}
                messages.append({"role": "user", "content": problem})
                continue
            if problem:   # still unsupported after the push-backs: deliver it, clearly flagged
                answer = f"{answer}\n\nUNVERIFIED: {problem}"
            yield {"final_answer": answer, "evidence_path": move.get("evidence_path", "")}
            return
        tool, args = move.get("action"), move.get("action_input", {}) or {}
        if tool not in tools:
            observation = f"Unknown tool {tool!r}. Use one of: {', '.join(tools)}."
            args = args if isinstance(args, dict) else {}
        else:
            if not isinstance(args, dict):   # e.g. "action_input": "data/data": give it to the 1st arg
                params = list(inspect.signature(tools[tool]).parameters)
                args = {params[0]: args} if params else {}
            try:
                observation = tools[tool](**args)
                tools_run += 1                 # only a tool that actually ran counts as evidence
                ran.add(tool)
            except Exception as e:
                observation = f"Tool error: {e}. Arguments are: {tool}{inspect.signature(tools[tool])}"
        seen += " " + str(observation)
        yield {"step": step, "thought": move.get("thought", ""),
               "action": tool, "args": args, "observation": observation}
        messages.append({"role": "user", "content": f"Observation:\n{observation}"})
    # Step limit reached. Do not throw away what was found: ask once for the best answer from the
    # observations so far, and run it through the same evidence check (flagged if it fails).
    if tools_run:
        messages.append({"role": "user", "content": "You have reached the step limit. Reply now with your "
                         "final_answer JSON, based only on the observations above. If they do not answer "
                         "the question, say so plainly."})
        move = parse_action(chat(client, model, messages))
        if move and "final_answer" in move:
            answer = move["final_answer"]
            problem = evidence_check(answer, tools_run, seen, cited=move.get("evidence_path", ""),
                                     ran=ran, tool_names=list(tools))
            if problem:
                answer = f"{as_text(answer)}\n\nUNVERIFIED: {problem}"
            yield {"final_answer": answer, "evidence_path": move.get("evidence_path", "")}
            return
    yield {"final_answer": None, "evidence_path": ""}


# ======================================================================= evidence check
# Limiting the tools to the evidence limits what the model can LOOK AT, not what it can SAY: a
# model can still write a plausible PID or file name from its training. So before an answer is
# accepted, the harness checks it: at least one tool must have run for this question, and every
# specific identifier in the answer must appear in something a tool actually returned (or in the
# question / earlier answers). Anything else is sent back to the model to confirm or remove.
MAX_PUSHBACKS = 2
EVIDENCE_PATTERNS = {
    "PID": r"\bPIDs?\s*[:#=]?\s*(\d{1,6})\b",
    "IP address": r"(?<![\d.])((?:\d{1,3}\.){3}\d{1,3})(?![\d.])",
    "memory address": r"\b(0x[0-9a-fA-F]{5,})\b",
    "file": r"\b([\w-]+\.(?:exe|dll|sys|apk|db|sqlite|xml|json|plist|log|txt|csv))\b",
}


def evidence_check(answer, tools_run, seen, cited="", ran=(), tool_names=()):
    """Return a push-back message if the answer is not backed by tool output, else None.
    cited / ran / tool_names: the answer's evidence line must not name a tool that never ran."""
    answer = as_text(answer)
    if tools_run == 0 and not answer.rstrip().endswith("?"):   # a question back to the user claims nothing
        return ("You answered without running any tool. Do not answer from memory or assumption: "
                "run the tool that shows this evidence, then answer only from its output.")
    seen_low = seen.lower()
    missing = []
    for kind, pattern in EVIDENCE_PATTERNS.items():
        for value in re.findall(pattern, str(answer), flags=re.I):
            found = (re.search(rf"(?<!\d){value}(?!\d)", seen) if kind == "PID"
                     else value.lower() in seen_low)
            if not found and f"{kind} {value}" not in missing:
                missing.append(f"{kind} {value}")
    if missing:
        return ("Your answer names " + ", ".join(missing[:8]) + ", which does not appear in any tool "
                "output you have seen. Run a tool that shows it, or remove it from your answer.")
    cited_text = as_text(cited)
    unrun = [t for t in tool_names if t not in ran and re.search(rf"\b{re.escape(t)}\b", cited_text)
             and not re.search(rf"\b{re.escape(t)}\b", seen)]
    if unrun:
        return ("Your evidence cites " + ", ".join(unrun) + ", which you have not run. Cite only the tools "
                "whose output you actually used, or run them first.")
    return None


def run_agent(question, client, model, tools, system_prompt=None, max_steps=12,
              on_step=None, history=None):
    """Drive the model think->act->observe until it answers.

    on_step(dict): optional callback for live UIs (gets each step dict, then the final dict).
    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    Returns the final move dict (with 'final_answer'/'evidence_path'), or None if it gave up.
    """
    final = None
    for ev in run_agent_iter(question, client, model, tools, system_prompt, max_steps, history):
        if on_step:
            on_step(ev)
        if "final_answer" in ev:
            final = ev if ev["final_answer"] is not None else None
    return final


def answer_text(move):
    """Format a final move as a human answer line with its citation."""
    if not move:
        return "I could not reach an answer within the step limit. Try rephrasing the question."
    cite = move.get("evidence_path") or "(no file cited)"
    return f"{move['final_answer']}\n\nSource: {cite}"


In [ ]:
%%writefile memkit.py
"""
memkit.py - Sectra Memory Analyst: a memory-forensics agent over Volatility 3.

THE IDEA
--------
Volatility already gives an examiner a list of plugins (pslist, pstree, malfind, ...). Here that
plugin list IS the agent's toolset. The analyst asks a question in plain English; the language model
picks ONE plugin and its arguments; our code runs `vol -r json -f <image> <plugin>`; the model reads
the result and either runs another plugin or answers, citing the plugin and the PID it relied on.

DESIGN RULES (the same evidence discipline as the rest of the course)
---------------------------------------------------------------------
- A fixed WHITELIST of read-only Volatility plugins. The model can only choose from it; nothing is
  passed to a shell, and arguments are validated (a PID must be a number).
- Every plugin runs ONCE per image and is cached; later questions reuse the cached result.
- Output is filtered (by PID or keyword) and capped, so the model is not flooded.
- A malfind hit is a LEAD, not proof. Every claim must cite the plugin and PID behind it.
- Volatility 3's network plugins do not support Windows XP; on XP, search_memory (vadregexscan) finds
  network indicators inside process memory instead.

Reuses the shared agent loop in agentkit (connect_model, run_agent, run_agent_iter).
"""
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import agentkit as ak

connect_model = ak.connect_model   # 'groq' (default; same free key as Labs 01-06), 'openrouter', 'ollama'


run_agent = ak.run_agent
run_agent_iter = ak.run_agent_iter
answer_text = ak.answer_text

MAX_ROWS = 40          # rows shown to the model per call
MAX_CHARS = 3500       # hard cap on one observation

# ------------------------------------------------------------------ the public sample image
# Cridex: a classic Windows XP memory image of a banking-trojan infection, widely used to teach
# memory forensics, with publicly documented ground truth. 40 MB download, 512 MB unpacked.
SAMPLE = {
    "name": "cridex",
    "url": "https://github.com/Divinemonk/memory_forensics_with_volatility/releases/download/mem_sample/cridexmemdump.zip",
    "zip_sha256": "6453d0986352f7d5287cb90eeb6b72bc7bc16b03e964bb5767d93f85c3a483b2",
    "image": "cridex.vmem",
    "image_md5": "7494f3b77db1525c1974f5380744ae46",
}


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_sample(dest="memory", force=False):
    """Download the public Cridex memory image, verify its hashes, and unzip it. Returns the image path."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    image = dest / SAMPLE["image"]
    if image.exists() and not force:
        print(f"Already have {image} ({image.stat().st_size // (1 << 20)} MB).")
        return str(image)
    zpath = dest / "cridexmemdump.zip"
    print("Downloading the Cridex sample (about 40 MB)...")
    try:
        import requests
        with requests.get(SAMPLE["url"], stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zpath, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
    except ImportError:
        import urllib.request
        try:
            urllib.request.urlretrieve(SAMPLE["url"], zpath)
        except Exception as e:
            raise RuntimeError(f"Download failed ({e}). Run `pip install requests` and try again.") from e
    got = _sha256(zpath)
    if got != SAMPLE["zip_sha256"]:
        raise RuntimeError(f"Download hash mismatch ({got}); the file may have changed. Not using it.")
    with zipfile.ZipFile(zpath) as z:
        z.extract(SAMPLE["image"], dest)
    zpath.unlink()
    md5 = _md5(image)
    if md5 != SAMPLE["image_md5"]:
        raise RuntimeError(f"{image} has MD5 {md5}, not the published {SAMPLE['image_md5']}. Not using it.")
    print(f"Verified {image}: MD5 {md5} (matches the published sample).")
    return str(image)


# ------------------------------------------------------------------ the toolset: whitelisted plugins
# name -> spec. vol: candidate plugin names (newest first; Volatility renames some between versions).
# pid: None (not used) / "optional" / "required". keyword: None / "optional" / "required".
PLUGINS = {
    "info": dict(vol=["windows.info"], cols=["Variable", "Value"],
                 doc="OS version, build, and capture time of the memory image. A good first step."),
    "pslist": dict(vol=["windows.pslist"], pid="optional",
                   cols=["PID", "PPID", "ImageFileName", "Threads", "Handles", "CreateTime", "ExitTime"],
                   doc="Running processes from the active process list (PID, parent PID, name, start time)."),
    "pstree": dict(vol=["windows.pstree"], tree=True,
                   cols=["PID", "PPID", "ImageFileName", "CreateTime"],
                   doc="Processes as a parent/child tree: shows which process launched which."),
    "psscan": dict(vol=["windows.psscan"], pid="optional",
                   cols=["PID", "PPID", "ImageFileName", "CreateTime", "ExitTime"],
                   doc="Scans raw memory for process objects, including exited or hidden ones pslist misses."),
    "psxview": dict(vol=["windows.malware.psxview", "windows.psxview"], pid="optional",
                    doc="Cross-checks several process lists; a process missing from a view may be hidden."),
    "cmdline": dict(vol=["windows.cmdline"], pid="optional", cols=["PID", "Process", "Args"],
                    doc="The command line each process was started with."),
    "dlllist": dict(vol=["windows.dlllist"], pid="required", keyword="optional",
                    cols=["PID", "Process", "Name", "Path"],
                    doc="DLLs loaded by ONE process (needs pid; optional keyword filters by name/path)."),
    "ldrmodules": dict(vol=["windows.malware.ldrmodules", "windows.ldrmodules"], pid="required",
                       doc="Modules mapped in ONE process and whether each loader list knows them (False = possibly unlinked/hidden). Needs pid."),
    "handles": dict(vol=["windows.handles"], pid="required", keyword="optional",
                    cols=["PID", "Process", "Type", "Name"],
                    doc="Open handles (files, registry keys, mutexes...) of ONE process (needs pid; optional keyword)."),
    "malfind": dict(vol=["windows.malware.malfind", "windows.malfind"], pid="optional",
                    doc="Memory regions that look like injected code (executable, private, often with a PE 'MZ' header). A hit is a LEAD, not proof."),
    "mutantscan": dict(vol=["windows.mutantscan"], keyword="optional",
                       doc="Named mutexes in memory; malware often creates distinctive ones (optional keyword filter)."),
    "svcscan": dict(vol=["windows.svcscan"], keyword="optional",
                    doc="Windows services: name, state, and binary path (optional keyword filter)."),
    "filescan": dict(vol=["windows.filescan"], keyword="required",
                     doc="File objects in memory. Needs a keyword (e.g. '.exe', 'Temp', 'Adobe') because the list is long."),
    "hivelist": dict(vol=["windows.registry.hivelist"], cols=["Offset", "FileFullPath"],
                     doc="Registry hives loaded in memory."),
    "getsids": dict(vol=["windows.getsids"], pid="optional", cols=["PID", "Process", "SID", "Name"],
                    doc="Which user and group SIDs a process runs as."),
    "envars": dict(vol=["windows.envars"], pid="required", keyword="optional",
                   cols=["PID", "Process", "Variable", "Value"],
                   doc="Environment variables of ONE process (needs pid)."),
    "netscan": dict(vol=["windows.netscan"], pid="optional",
                    doc="Network connections and listening sockets (Windows Vista and later ONLY; on XP use search_memory)."),
}

_NOISY = {"__children", "File output", "Hexdump", "Disasm", "Notes"}
_HEXCOLS = {"Offset", "Offset(V)", "Offset(P)", "Base", "Start VPN", "End VPN", "Physical Address"}


def _ldr_note(rows):
    """State plainly the rows a reader must not miss: memory in NONE of the three loader lists
    (InLoad, InInit, InMem all False). That is a fact from the output, not a verdict."""
    def _false(v):
        return str(v).strip().lower() == "false"
    odd = [r for r in rows if all(_false(r.get(k)) for k in ("InLoad", "InInit", "InMem"))]
    if not odd:
        return "NOTE: every module here is in at least one loader list.\n"
    def _addr(v):
        return hex(v) if isinstance(v, int) else str(v)
    items = ", ".join(f"{_addr(r.get('Base'))} (mapped file: {r.get('MappedPath') or '-'})" for r in odd)
    return (f"NOTE: {len(odd)} region(s) appear in NONE of the three loader lists (InLoad, InInit and InMem "
            f"all False): {items}. A region the loader does not know about, with no file behind "
            "it, is a classic sign of injected code; corroborate with malfind.\n")


def _vol_version():
    """The installed Volatility 3 version (part of the plugin-check cache key)."""
    try:
        from importlib.metadata import version
        return version("volatility3")
    except Exception:
        return "unknown"


def _vol_cmd():
    """Find the Volatility 3 command-line entry point installed with `pip install volatility3`."""
    here = Path(sys.executable).with_name("vol")
    if here.exists():
        return str(here)
    found = shutil.which("vol")
    if found:
        return found
    raise RuntimeError("Volatility 3 is not installed. Run: pip install volatility3")


def _flatten(rows, depth=0):
    """Flatten Volatility's nested __children (pstree) into rows with a depth marker."""
    out = []
    for r in rows:
        r = dict(r)
        if "PID" not in r and "Pid" in r:      # e.g. ldrmodules names its column Pid, not PID
            r["PID"] = r["Pid"]
        kids = r.pop("__children", []) or []
        r["_depth"] = depth
        out.append(r)
        out += _flatten(kids, depth + 1)
    return out


def _fmt_cell(col, val):
    if val is None or val == "None":
        return "-"
    if col in _HEXCOLS and isinstance(val, int):
        return hex(val)
    s = str(val).replace("\t", " ").replace("\n", " ")
    return s if len(s) <= 120 else s[:117] + "..."


class VolRunner:
    """Runs whitelisted plugins once per image and caches the JSON rows."""

    def __init__(self, image, cache_dir=".volcache", ran=None):
        self.image = str(image)
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.mem = {}
        self.ran = ran if ran is not None else []   # audit trail: every Volatility command used (shareable)

    def _log(self, plugin, extra, label):
        entry = "vol -f <image> " + " ".join([plugin, label or " ".join(extra)]).strip()
        if entry not in self.ran:
            self.ran.append(entry)

    def _key(self, plugin, extra):
        h = hashlib.sha1(f"{self.image}|{plugin}|{extra}".encode()).hexdigest()[:16]
        return self.cache_dir / f"{plugin.replace('.', '_')}_{h}.json"

    def run(self, candidates, extra=(), label=None):
        """Run the first plugin name this Volatility version knows; return (rows, plugin_used, error).
        label: a short human-readable stand-in for long arguments in the audit trail."""
        last_err = ""
        for plugin in candidates:
            key = self._key(plugin, "|".join(extra))
            if key in self.mem:
                self._log(plugin, extra, label)
                return self.mem[key], plugin, None
            if key.exists():
                rows = json.loads(key.read_text())
                self.mem[key] = rows
                self._log(plugin, extra, label)
                return rows, plugin, None
            cmd = [_vol_cmd(), "-q", "-r", "json", "-f", self.image, plugin, *extra]
            # no colour codes: notebooks make Python print coloured tracebacks, which hide the error text
            env = dict(os.environ, NO_COLOR="1", PYTHON_COLORS="0", TERM="dumb")
            p = subprocess.run(cmd, capture_output=True, text=True, timeout=900, env=env)
            if p.returncode == 0 and p.stdout.strip().startswith("["):
                rows = _flatten(json.loads(p.stdout))
                key.write_text(json.dumps(rows))
                self.mem[key] = rows
                self._log(plugin, extra, label)
                return rows, plugin, None
            raw = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", p.stderr or p.stdout or "")   # strip any ANSI codes
            err = raw.strip().splitlines()
            last_err = " | ".join(err[-25:]) if err else f"exit code {p.returncode}"
            if "invalid choice" not in last_err:
                break                       # a real failure, not just an unknown plugin name
        return None, candidates[0], last_err


def _table(rows, cols=None, limit=MAX_ROWS, tree=False):
    if not rows:
        return "(no rows)"
    cols = cols or [c for c in rows[0] if c not in _NOISY and not c.startswith("_")]
    lines = ["\t".join(cols)]
    for r in rows[:limit]:
        cells = [_fmt_cell(c, r.get(c)) for c in cols]
        if tree and r.get("_depth"):
            cells[0] = "." * r["_depth"] + cells[0]
        lines.append("\t".join(cells))
    if len(rows) > limit:
        lines.append(f"... {len(rows) - limit} more rows not shown. Narrow it with pid= or keyword=.")
    return "\n".join(lines)


def _malfind_view(rows):
    """malfind rows are huge (hexdump + disassembly); keep what an examiner reads first."""
    if not rows:
        return "(no suspicious regions found)"
    lines = ["PID\tProcess\tStart\tProtection\tPE header?\tfirst bytes"]
    for r in rows[:MAX_ROWS]:
        dump = (r.get("Hexdump") or "").replace("\n", " ")
        first = " ".join(dump.split()[:16])
        mz = "YES (MZ - a PE file in memory)" if "4d 5a 90 00" in dump[:200] else "no"
        lines.append(f"{r.get('PID')}\t{r.get('Process')}\t{_fmt_cell('Start VPN', r.get('Start VPN'))}\t"
                     f"{r.get('Protection')}\t{mz}\t{first}")
    if len(rows) > MAX_ROWS:
        lines.append(f"... {len(rows) - MAX_ROWS} more regions. Narrow with pid=.")
    return "\n".join(lines)


def _clip(text):
    return text if len(text) <= MAX_CHARS else text[:MAX_CHARS] + "\n...[truncated - narrow with pid= or keyword=]"


# ------------------------------------------------------------------ plugin compatibility check
def _short_reason(err):
    """Volatility's own error message (the last 'XxxError: ...'), not the traceback's source lines."""
    found = [m for m in re.findall(r"(?:Error|Exception):\s*([^|]+)", err or "") if "{" not in m]
    if found:
        return found[-1].strip()[:90]
    parts = [p.strip() for p in (err or "").split("|") if p.strip() and "{" not in p]
    return (parts[-1] if parts else "failed")[:90]


def check_plugins(image, cache_dir=".volcache", verbose=True):
    """Run every whitelisted plugin ONCE on this image and report which ones return data.

    Not every plugin supports every image (for example, Volatility 3's netscan does not support
    Windows XP). The lab gives the agent only the plugins that work on THIS image, so it never wastes
    a step on one that cannot answer. Results are cached, and the runs also warm the plugin cache, so
    later questions are instant. Returns {name: {"status": ok|empty|unsupported, "rows": n, "note": ...}}."""
    vr = VolRunner(image, cache_dir)
    tag = hashlib.sha1(f"{Path(image).resolve()}|{os.path.getsize(image)}|{_vol_version()}".encode()).hexdigest()[:16]
    key = Path(cache_dir) / f"compat_{tag}.json"
    # Only results that WORKED are cached. A failure is re-checked each time, so a one-off problem
    # (a symbol download that timed out, say) never removes a plugin for good.
    cached = json.loads(key.read_text()) if key.exists() else {}
    result = {n: r for n, r in cached.items() if n in PLUGINS and r.get("status") in ("ok", "empty")}
    todo = [n for n in PLUGINS if n not in result]
    if todo:
        if verbose:
            print(f"Checking {len(todo)} plugins against {Path(image).name} (working results are cached)...")
        for name in todo:
            spec = PLUGINS[name]
            rows, used, err = vr.run(spec["vol"])
            if err:
                status, note = "unsupported", _short_reason(err)
            elif not rows:
                status, note = "empty", "ran, but returned no rows on this image"
            else:
                status, note = "ok", f"{len(rows)} rows"
            result[name] = {"status": status, "plugin": used, "rows": len(rows or []), "note": note}
            if verbose:
                print(f"  {'OK ' if status == 'ok' else '-- '} {name:11} {note}")
        key.parent.mkdir(parents=True, exist_ok=True)
        key.write_text(json.dumps({n: r for n, r in result.items() if r["status"] in ("ok", "empty")}))
        result = {n: result[n] for n in PLUGINS}   # keep the whitelist order
    if verbose:
        ok = [n for n, r in result.items() if r["status"] == "ok"]
        print(f"{len(ok)} of {len(result)} plugins work on this image and become the agent's tools.")
        for n, r in result.items():
            if r["status"] != "ok":
                print(f"  left out: {n} ({r['status']}: {r['note']})")
    return result


# ------------------------------------------------------------------ build the agent's tools
def make_memory_tools(image, case_dir="case", cache_dir=".volcache", findings=None, ran=None,
                      compatible_only=False, compat=None):
    """Return {tool_name: function}. One tool per whitelisted plugin, plus search_memory, the case brief,
    and the findings log. `findings` collects recorded findings; `ran` collects the Volatility commands
    used (pass your own lists to keep them across calls, e.g. one chat session).
    compatible_only=True keeps only the plugins check_plugins() found working on this image."""
    vr = VolRunner(image, cache_dir, ran=ran)
    if compatible_only:
        compat = compat or check_plugins(image, cache_dir, verbose=False)
    findings = findings if findings is not None else []
    case_dir = Path(case_dir)
    tools = {}

    def make_plugin_tool(name, spec):
        def tool(pid=None, keyword=None):
            if pid in ("", "None"):
                pid = None
            if pid is not None:
                try:
                    pid = int(pid)
                except (TypeError, ValueError):
                    return f"pid must be a number, got {pid!r}."
            if spec.get("pid") == "required" and pid is None:
                return f"{name} needs a pid (one process). Find candidate PIDs with pslist or pstree first."
            if spec.get("keyword") == "required" and not keyword:
                return f"{name} needs a keyword (for example '.exe' or 'Temp') because its output is long."
            rows, used, err = vr.run(spec["vol"])
            if err:
                if name == "netscan":
                    return ("netscan failed on this image. Volatility 3's network plugins only support Windows "
                            "Vista and later; on Windows XP use search_memory(kind='ip') to find network "
                            f"indicators inside process memory. (Volatility said: {_short_reason(err)})")
                return f"{used} failed: {_short_reason(err)}"
            if name == "info":                 # drop the local symbol-cache path (noise, not evidence)
                rows = [r for r in rows if r.get("Variable") != "Symbols"]
            if pid is not None and spec.get("pid"):
                rows = [r for r in rows if str(r.get("PID", r.get("Pid"))) == str(pid)]
            if keyword:
                k = str(keyword).lower()
                rows = [r for r in rows if k in " ".join(str(v) for v in r.values()).lower()]
            body = _malfind_view(rows) if name == "malfind" else _table(rows, spec.get("cols"), tree=spec.get("tree"))
            if name == "ldrmodules":
                body = _ldr_note(rows) + body
            shown_pid = f" --pid {pid}" if pid is not None and spec.get("pid") else ""
            shown_kw = f" | filter: {keyword}" if keyword else ""
            head = f"[{used}{shown_pid}{shown_kw}] {len(rows)} rows\n"
            return _clip(head + body)

        args = []
        if spec.get("pid"):
            args.append("pid" + ("" if spec["pid"] == "required" else "=optional"))
        if spec.get("keyword"):
            args.append("keyword" + ("" if spec["keyword"] == "required" else "=optional"))
        tool.__doc__ = f"({', '.join(args)}) {spec['doc']}" if args else spec["doc"]
        tool.__name__ = name
        return tool

    for name, spec in PLUGINS.items():
        if compat and compat.get(name, {}).get("status") != "ok":
            continue                         # not working on this image: the agent never sees it
        tools[name] = make_plugin_tool(name, spec)

    def search_memory(kind="ip", text="", pid=None):
        """(kind='ip'|'url'|'text', text=optional, pid=optional) Search inside PROCESS memory (windows.vadregexscan) for network indicators (ip = IP:port), URLs, or a literal text, and show which process holds each hit."""
        patterns = {
            "ip": r"(?:25[0-5]|2[0-4][0-9]|1?[0-9]?[0-9])(?:\.(?:25[0-5]|2[0-4][0-9]|1?[0-9]?[0-9])){3}:[0-9]{2,5}",
            "url": r"https?://[A-Za-z0-9./_?=&%-]{4,80}",
        }
        if text:
            pattern, label = re.escape(str(text)), f"text {text!r}"
        elif kind in patterns:
            pattern, label = patterns[kind], kind
        else:
            return "kind must be 'ip' or 'url', or pass text='...' to search for a literal string."
        extra = ["--pattern", pattern]
        if pid not in (None, "", "None"):
            try:
                extra += ["--pid", str(int(pid))]
            except (TypeError, ValueError):
                return f"pid must be a number, got {pid!r}."
        rows, used, err = vr.run(["windows.vadregexscan"], tuple(extra), label=f"--pattern <{label}>")
        if err:
            return f"search failed: {_short_reason(err)}"
        seen = {}
        for r in rows:
            t = str(r.get("Text", ""))
            k = (r.get("PID"), r.get("Process"), t)
            seen[k] = seen.get(k, 0) + 1
        if not seen:
            return f"[windows.vadregexscan] no {label} found in process memory."
        lines = [f"[windows.vadregexscan pattern={label}] {len(rows)} hits, {len(seen)} distinct (PID / process / text / count):"]
        for (p, proc, t), n in sorted(seen.items(), key=lambda kv: -kv[1])[:30]:
            lines.append(f"{p}\t{proc}\t{t}\tx{n}")
        return _clip("\n".join(lines))

    tools["search_memory"] = search_memory

    def case_brief():
        """The case brief: what happened, what was collected, and what the investigator needs answered."""
        p = case_dir / "case_brief.md"
        return p.read_text() if p.exists() else "No case brief found (case/case_brief.md)."

    def record_finding(finding, evidence):
        """(finding, evidence) Write a confirmed finding to the case log, with the plugin + PID that supports it."""
        if not finding or not evidence:
            return "A finding needs both the finding text and the evidence (plugin + PID) behind it."
        findings.append({"finding": str(finding), "evidence": str(evidence)})
        return f"Recorded finding #{len(findings)}."

    def findings_report():
        """Produce the findings report from everything recorded in the case log so far."""
        lines = ["FINDINGS REPORT (memory analysis)", "=" * 34,
                 f"Image: {Path(vr.image).name}", ""]
        if not findings:
            lines.append("(No findings recorded yet. Use record_finding as you confirm each one.)")
        for i, f in enumerate(findings, 1):
            lines.append(f"{i}. {f['finding']}\n   Evidence: {f['evidence']}")
        lines += ["", "Volatility commands run:"] + [f"  {c}" for c in vr.ran] + [
            "", "Note: findings are bounded by memory evidence. A malfind hit or a string in memory is a lead; "
            "confirm with disk, network, and malware analysis before attributing intent."]
        return "\n".join(lines)

    tools["case_brief"] = case_brief
    tools["record_finding"] = record_finding
    tools["findings_report"] = findings_report
    tools["_runner"] = vr               # not a tool; exposed for notebooks (stripped from the prompt)
    tools["_compat"] = compat or {}     # not a tool; the compatibility check, summarised in the prompt
    return tools


def agent_tools(tools):
    """The tools the model may call (drops private helpers like _runner)."""
    return {k: v for k, v in tools.items() if not k.startswith("_")}


def tool_descriptions(tools):
    return "\n".join(f"- {n}: {(fn.__doc__ or '').strip().splitlines()[0]}" for n, fn in agent_tools(tools).items())


def _compat_note(tools):
    compat = tools.get("_compat") or {}
    if not compat:
        return ""
    left = [f"{n} ({r['status']}: {r['note']})" for n, r in compat.items() if r["status"] != "ok"]
    note = "PLUGIN CHECK: every plugin tool above was tested on this image and returns data."
    if left:
        note += (" Left out: " + "; ".join(left) + ". An 'empty' plugin found nothing on this image, which is "
                 "itself a result you may report. For network indicators, use search_memory.")
    return note + "\n\n"


def build_memory_system_prompt(tools, case_name="the case"):
    return (
        f"You are Sectra, a memory-forensics analyst working {case_name}. You investigate a Windows memory "
        "image with Volatility 3. You cannot see memory directly: you learn ONLY by running the plugins below, "
        "and you must base every statement on what they return.\n\n"
        "TOOLS (each Volatility plugin is a tool; use exactly ONE per step):\n" + tool_descriptions(tools) + "\n\n"
        + _compat_note(tools) +
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To run a plugin:\n"
        '  {"thought": "<one short sentence>", "action": "pstree", "action_input": {}}\n'
        '  {"thought": "<one short sentence>", "action": "dlllist", "action_input": {"pid": 1640}}\n'
        "When you can answer the investigator:\n"
        '  {"thought": "<one short sentence>", "final_answer": "<answer>", "evidence_path": "<plugin + PID you relied on>"}\n\n'
        "HOW TO WORK THE CASE:\n"
        "1. Pick the plugin that answers the question most directly (process questions: pslist/pstree; code "
        "injection: malfind; what a process loaded: dlllist/ldrmodules; network on Windows XP: search_memory).\n"
        "2. Read the result, then corroborate before concluding: a suspicious finding should be supported by "
        "a second plugin (for example malfind plus pstree plus cmdline).\n"
        "3. When something is confirmed, record it with record_finding (finding + plugin + PID).\n\n"
        "RULES: cite the plugin and PID in evidence_path; a malfind hit is a LEAD, not proof (system processes "
        "like csrss.exe and winlogon.exe often show benign hits); never name a malware family or claim intent "
        "unless the plugin output supports it; say plainly what memory cannot establish; keep thoughts to one "
        "sentence and answers short. /no_think"
    )


def quick_agent(question, client, model, tools, history=None, **kw):
    prompt = build_memory_system_prompt(tools)
    return run_agent(question, client, model, agent_tools(tools), prompt, history=history, **kw)


In [ ]:
%%writefile case/case_brief.md
# Case MF-2026-001: Strange behaviour on an accounts workstation (training case)

## What happened
A clerk in the accounts office of a small firm reported that their Windows XP workstation became slow
and that their online-banking session "behaved strangely" shortly after they opened a PDF attachment.
The workstation was not shut down. IT captured a copy of its physical memory (RAM) before isolating it.

## What was collected
- One memory image: `cridex.vmem` (512 MB, captured 2012-07-22).
- No disk image, no network capture. Memory is the only evidence in this case.

## What the investigator needs answered
1. What operating system is this, and when was memory captured?
2. Which processes were running, and which process launched which?
3. Is any process unusual for this system? Why?
4. Is there evidence of code injected into a running process? Which ones?
5. Are there any network indicators (addresses the machine was talking to)? Which process holds them?
6. What can the memory evidence NOT tell us, and what would you collect next?

## Ground rules
- Every finding must cite the Volatility plugin and the PID it came from.
- A suspicious result is a lead. Corroborate it with a second plugin before you call it a finding.
- This is a training case. The memory image is the public Cridex teaching sample; the story above is a
  fictional framing for the class.


### 2. Get the memory image

The notebook downloads the public Cridex sample and verifies its hash, so everyone analyses the
same bytes. To use **your own** image, set `MEMORY_IMAGE` to its path instead (Windows images
work best; on Windows Vista and later, the network plugin `netscan` also works).

In [ ]:
import memkit as mk

# Downloads the public Cridex sample (about 40 MB), checks its SHA-256, and unzips it (512 MB).
MEMORY_IMAGE = mk.fetch_sample('memory')

# Using your own memory image instead? Upload it and point to it, for example:
# MEMORY_IMAGE = '/content/my_memory.raw'
print('Image:', MEMORY_IMAGE)

### 3. First, the old way: run Volatility yourself

This is what an examiner types by hand. You have to know **which** plugin answers **which**
question, then read the raw table. The first run downloads Windows symbol files (about a minute).

In [ ]:
VOL = mk._vol_cmd()
!{VOL} -q -f {MEMORY_IMAGE} windows.info

In [ ]:
!{VOL} -q -f {MEMORY_IMAGE} windows.pstree

### 4. Check which plugins work on this image

Not every plugin supports every image (for example, Volatility 3's `netscan` does not support
Windows XP). So before the agent starts, the lab runs every whitelisted plugin once and keeps only
the ones that return data. This takes a minute or two once; the results are cached, so every
later answer is fast. The agent never wastes a step on a plugin that cannot answer.

In [ ]:
COMPAT = mk.check_plugins(MEMORY_IMAGE)

Now let the agent choose the plugins, from the ones that work here.

### 5. Connect the model and wire up Sectra

In [ ]:
import os, getpass
PROVIDER = 'groq'   # free key at console.groq.com  |  or 'openrouter' or 'ollama' (local Qwen)
if PROVIDER == 'groq' and not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()
if PROVIDER == 'openrouter' and not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (hidden): ').strip()
client, MODEL = mk.connect_model(PROVIDER)

FINDINGS, RAN = [], []          # the case log and the Volatility commands used
ALL = mk.make_memory_tools(MEMORY_IMAGE, case_dir='case', findings=FINDINGS, ran=RAN,
                           compatible_only=True, compat=COMPAT)   # only plugins that work on this image
TOOLS = mk.agent_tools(ALL)
PROMPT = mk.build_memory_system_prompt(ALL, 'case MF-2026-001')
conversation = []               # Sectra remembers the case across talk() calls
print('Model:', MODEL)
print('Toolset (Volatility plugins + case tools):', ', '.join(TOOLS))

### 6. Ask Sectra

`talk()` prints each plugin Sectra runs and its answer, with the evidence it cites.

In [ ]:
def talk(message):
    print('You:', message, '\n')
    def show(ev):
        if 'final_answer' in ev:
            print('\nSectra:', ev['final_answer'])
            if ev.get('evidence_path'): print('   evidence:', ev['evidence_path'])
        else:
            th = (ev.get('thought') or '').strip()
            print(f"  [{ev['step']}] {th}")
            print(f"      -> {ev['action']}({ev['args']})  {str(ev['observation']).splitlines()[0][:80]}")
    move = mk.run_agent(message, client, MODEL, TOOLS, PROMPT, on_step=show, history=conversation, max_steps=10)
    conversation.append((message, move['final_answer'] if move else '(no answer)'))
    return move

_ = talk('What operating system is this, and when was memory captured?')

### 7. Work the case

These follow the questions in the case brief. Watch which plugin Sectra chooses for each one, and
whether it corroborates a suspicious hit with a second plugin before calling it a finding.

In [ ]:
_ = talk('Show me the process tree. Is anything unusual?')

In [ ]:
_ = talk('Is there evidence of injected code in any process? Which hits matter and which are likely benign?')

In [ ]:
_ = talk('What command line did PID 1640 start with, and what launched it?')

In [ ]:
_ = talk('Are there any network indicators in memory, and which process holds them?')

In [ ]:
_ = talk('Record what you have confirmed so far, each with its plugin and PID.')

In [ ]:
print(ALL['findings_report']())

### 8. Check the AI against the ground truth

Open the [answer key](https://github.com/danjethh/sectratek-AI-Lab/blob/main/lab10_llm_memory_forensics_volatility/answer_key.md) and compare. Questions to discuss:

- Did Sectra flag the **csrss.exe / winlogon.exe** malfind hits as findings, or recognise them as benign?
- Did it **name the malware family** (Cridex)? Is that supported by memory alone, or by outside knowledge?
- Which claims did it corroborate with a second plugin, and which rest on one plugin only?
- What can memory **not** tell you here, and what would you collect next?

### 9. Your turn

- Ask your own questions: *"Which DLLs did PID 1640 load?"*, *"Any unusual mutexes?"*, *"What services exist?"*
- Point `MEMORY_IMAGE` at another image (for example a Windows 7/10 CTF image), then re-run the plugin
  check (step 4) and the wiring cell (step 5) so the tools describe the new image, and work a new case.
- `print('\n'.join(RAN))` shows every Volatility command the agent used.

In [ ]:
print('\n'.join(RAN))